#**LangChain**

LangChain is a framework for developing applications powered by language models.

- GitHub: https://github.com/hwchase17/langchain
- Docs: https://python.langchain.com/v0.2/docs/introduction/

### Overview:
- Installation
- LLMs
- Prompt Templates
- Chains
- Agents and Tools
- Memory
- Document Loaders
- Indexes

#**01: Installation**

In [8]:
!pip install langchain langchain_community
!pip install -U langchain-core langchain-huggingface

  Using cached langchain_core-0.3.84-py3-none-any.whl.metadata (3.2 kB)
Using cached langchain_core-0.3.84-py3-none-any.whl (459 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.0
    Uninstalling langchain-core-1.3.0:
      Successfully uninstalled langchain-core-1.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.4 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.3.84 which is incompatible.
langchain-classic 1.0.4 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.8 which is incompatible.
langchain-huggingface 1.2.2 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.3.84 which is incompatible.
langgraph-prebuilt 1.0.9 requires langchain-core>=1.0.0, but you have langchain-core 0.3.84 which is incompatible.


  Using cached langchain_core-1.3.0-py3-none-any.whl.metadata (4.4 kB)
Using cached langchain_core-1.3.0-py3-none-any.whl (515 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.84
    Uninstalling langchain-core-0.3.84:
      Successfully uninstalled langchain-core-0.3.84
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.4 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.8 which is incompatible.
langchain-text-splitters 0.3.8 requires langchain-core<1.0.0,>=0.3.51, but you have langchain-core 1.3.0 which is incompatible.
langchain-openai 0.3.17 requires langchain-core<1.0.0,>=0.3.59, but you have langchain-core 1.3.0 which is incompatible.
langchain 0.3.25 requires langchain-core<1.0.0,>=0.3.58, but you have langchain-core 1.3.0 which is incompatible.
lang

#**02: Setup the Environment**

In [9]:
import os
from getpass import getpass

In [10]:
os.environ['OPENAI_API_KEY'] =  getpass("OpenAI API Key: ")

In [11]:
os.environ["HF_TOKEN"] = getpass("HuggingFaceHub API Key: ")

##**03: Large Language Models**

The basic building block of LangChain is a Large Language Model which takes text as input and generates more text

Suppose we want to generate a company name based on the company description, so we will first initialize an OpenAI wrapper. In this case, since we want the output to be more random, we will intialize our model with high temprature.

The temperature parameter adjusts the randomness of the output. Higher values like 0.7 will make the output more random, while lower values like 0.2 will make it more focused and deterministic.

temperature value--> how creative we want our model to be

0 ---> temperature it means model is  very safe it is not taking any bets.

1 --> it will take risk it might generate wrong output but it is very creative

A generic interface for all LLMs. See all LLM providers: https://python.langchain.com/en/latest/modules/models/llms/integrations.html

#**Open AI**

#**Example 1**

In [12]:
from langchain_openai import OpenAI
llm = OpenAI(temperature=0.9)

And now we will pass in text and get  predictions

In [13]:
text="What would be a good company name for a company that makes colorful socks?"

In [14]:
llm.invoke(text)

'\n\n"Rainbow Footwear Co." or "Colorful Soles Co."'

In [15]:
llm.invoke(text)

'\n"Rainbow Soles"'

#**Example 2**

In [16]:
llm.invoke("I want to open a restaurant for Chinese food. Suggest a fency name for this.")


'\n\n"Dragon Palace" '

In [17]:
llm.invoke("I want to open a restaurant for Chinese food. Suggest a fency name for this.")

'\n\n"Dragon\'s Palace"'

#**Hugging Face**

#**Example 1**

In [18]:
# https://huggingface.co/google/flan-t5-xl

from langchain_huggingface import HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="google/flan-t5-base",
    temperature=0.1,
    max_new_tokens=64
)


In [19]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")


In [20]:
llm.invoke("translate English to German: How old are you?")

StopIteration: 

#**Example 2**

In [ ]:
from langchain import HuggingFaceHub

llm = HuggingFaceHub(repo_id="google/flan-t5-large", model_kwargs={"temperature":0, "max_length":64})
# name = llm.predict("I want to open a restaurant for Chinese food. Suggest a fency name for this.")
name = llm.predict("I want to open a restaurant for Indian food. Suggest a fency name for this.")
print(name)

/tmp/ipykernel_109526/2895970287.py:3: LangChainDeprecationWarning: The class `HuggingFaceHub` was deprecated in LangChain 0.0.21 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEndpoint``.
  llm = HuggingFaceHub(repo_id="google/flan-t5-large", model_kwargs={"temperature":0, "max_length":64})


ValidationError: 1 validation error for HuggingFaceHub
  Value error, Did not find huggingfacehub_api_token, please add an environment variable `HUGGINGFACEHUB_API_TOKEN` which contains it, or pass `huggingfacehub_api_token` as a named parameter. [type=value_error, input_value={'repo_id': 'google/flan-...acehub_api_token': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

##**04: Prompt Templates**

Currently in the above applications we are writing an entire prompt, if you are creating a user directed application then this is not an ideal case

LangChain faciliates prompt management and optimization.

Normally when you use an LLM in an application, you are not sending user input directly to the LLM. Instead, you need to take the user input and construct a prompt, and only then send that to the LLM.

In many Large Language Model applications we donot pass the user input directly to the Large Language Model, we add the user input to a large piece of text called prompt template

#**Example 1**

In [ ]:
from langchain.prompts import PromptTemplate

prompt_template_name = PromptTemplate(
    input_variables =['cuisine'],
    template = "I want to open a restaurant for {cuisine} food. Suggest a fency name for this."
)
p = prompt_template_name.format(cuisine="indian")
print(p)

I want to open a restaurant for indian food. Suggest a fency name for this.


#**Example 2**

In [ ]:
from langchain.prompts import PromptTemplate
prompt = PromptTemplate.from_template("What is a good name for a company that makes {product}")
prompt.format(product="colorful socks")

'What is a good name for a company that makes colorful socks'

##**05: Chains**

Combine LLMs and Prompts in multi-step workflows

Now as we have the  **model**:


  llm = OpenAI(temperature=0.9)


and the **Prompt Template**:

prompt = PromptTemplate.from_template("What is a good name for a company that makes {product}")


prompt.format(product="colorful socks")


Now using Chains we will link together model and the PromptTemplate and other Chains

The simplest and most common type of Chain is LLMChain, which passes the input first to Prompt Template and then to Large Language Model

LLMChain is responsible to execute the PromptTemplate, For every PromptTemplate we will specifically have an LLMChain

#**Example 1**

In [ ]:
from langchain.llms import OpenAI

llm = OpenAI(temperature=0.9)

In [ ]:
from langchain.prompts import PromptTemplate
prompt = PromptTemplate.from_template("What is a good name for a company that makes {product}")
prompt.format(product="colorful socks")

'What is a good name for a company that makes colorful socks'

Whatever input text i am giving that will get assigned to this particular variable that is **product**

In [ ]:
from langchain.chains import LLMChain

chain = LLMChain(llm=llm, prompt=prompt)
response= chain.run("colorful socks")
print(response)



"Rainbow Threads" or "ChromaSocks"


#**Example 2**

In [ ]:
from langchain.llms import OpenAI

llm = OpenAI(temperature=0.9)

In [ ]:
from langchain.prompts import PromptTemplate

prompt_template_name = PromptTemplate(
    input_variables =['cuisine'],
    template = "I want to open a restaurant for {cuisine} food. Suggest a fency name for this."
)

In [ ]:
from langchain.chains import LLMChain

chain = LLMChain(llm=llm, prompt=prompt_template_name)
response=chain.run("Mexican")
print(response)



"El Sabor de México" (The Flavor of Mexico)


In [ ]:
chain = LLMChain(llm=llm, prompt=prompt_template_name, verbose=True)
response=chain.run("Mexican")
print(response)

Prompt after formatting:
I want to open a restaurant for Mexican food. Suggest a fency name for this.

> Finished chain.


"La Casa de Sabores" (The House of Flavors)


**Can we combine Multiple PromptTemplates, We will try to combine Multiple PromptTemplates**

**The output from the first PromptTemplate is passed to the next PromptTemplate as input**

#**To combine the Chain and  to set a sequence for that we use SimpleSequentialChain**

##**Simple Sequential Chain**

In [ ]:
llm = OpenAI(temperature=0.6)

prompt_template_name = PromptTemplate(
    input_variables =['cuisine'],
    template = "I want to open a restaurant for {cuisine} food. Suggest a fency name for this."
)

name_chain =LLMChain(llm=llm, prompt=prompt_template_name)

prompt_template_items = PromptTemplate(
    input_variables = ['restaurant_name'],
    template="""Suggest some menu items for {restaurant_name}"""
)

food_items_chain = LLMChain(llm=llm, prompt=prompt_template_items)

In [ ]:
from langchain.chains import SimpleSequentialChain
chain = SimpleSequentialChain(chains = [name_chain, food_items_chain])

content = chain.run("indian")
print(content)



1. Spicy Szechuan Chicken
2. Hot and Sour Soup
3. Kung Pao Shrimp
4. Mapo Tofu
5. Spicy Beef and Broccoli
6. General Tso's Chicken
7. Dan Dan Noodles
8. Spicy Garlic Eggplant
9. Firecracker Shrimp
10. Spicy Sichuan Green Beans
11. Five Spice Pork Belly
12. Spicy Ma Po Tofu
13. Szechuan Peppercorn Beef
14. Spicy Hunan Tofu
15. Spicy Stir-Fried Cabbage
16. Spicy Chili Garlic Shrimp
17. Spicy Kung Pao Cauliflower
18. Szechuan Hot Pot
19. Spicy Szechuan Noodles
20. Fried Wontons with Sweet and Spicy Sauce.


**There is a issue with SimpleSequentialChain it only shows last input information**

#**To show the entire information i will use SequentialChain**

##**Sequential Chain**

In [ ]:
llm = OpenAI(temperature=0.7)

prompt_template_name = PromptTemplate(
    input_variables =['cuisine'],
    template = "I want to open a restaurant for {cuisine} food. Suggest a fency name for this."
)

name_chain =LLMChain(llm=llm, prompt=prompt_template_name, output_key="restaurant_name")

In [ ]:
llm = OpenAI(temperature=0.7)

prompt_template_items = PromptTemplate(
    input_variables = ['restaurant_name'],
    template="Suggest some menu items for {restaurant_name}."
)

food_items_chain =LLMChain(llm=llm, prompt=prompt_template_items, output_key="menu_items")

In [ ]:
from langchain.chains import SequentialChain

chain = SequentialChain(
    chains = [name_chain, food_items_chain],
    input_variables = ['cuisine'],
    output_variables = ['restaurant_name', "menu_items"]
)

In [ ]:
print(chain({"cuisine": "indian"}))

{'cuisine': 'indian', 'restaurant_name': '\n\n"Spice Palace"', 'menu_items': '\n\n1. Spicy Chicken Tikka Masala\n2. Vegetable Samosas\n3. Lamb Vindaloo\n4. Chana Masala (chickpea curry)\n5. Tandoori Grilled Shrimp\n6. Aloo Gobi (potato and cauliflower curry)\n7. Garlic Naan bread\n8. Mango Lassi (yogurt drink)\n9. Palak Paneer (spinach and cheese curry)\n10. Chicken Biryani (spiced rice dish)\n11. Masala Dosa (crispy rice crepe filled with spiced potatoes)\n12. Vegetable Korma (mixed vegetable curry)\n13. Gulab Jamun (fried dough balls in sweet syrup)\n14. Tandoori Chicken (marinated grilled chicken)\n15. Butter Chicken (creamy tomato-based curry)'}


##**06. Agents and Tools**

Agents involve an LLM making decisions about which Actions to take, taking that Action, seeing an Observation, and repeating that until done.


When used correctly agents can be extremely powerful. In order to load agents, you should understand the following concepts:

- Tool: A function that performs a specific duty. This can be things like: Google Search, Database lookup, Python REPL, other chains.
- LLM: The language model powering the agent.
- Agent: The agent to use.


Agent is a very powerful concept in LangChain

For example I have to travel from Dubai to Canada, I type this in ChatGPT



---> Give me  two flight options from Dubai to Canada on September 1, 2024 | ChatGPT will not be able to answer because has knowledge till
September 2021



ChatGPT plus has Expedia Plugin, if we enable this plugin it will go to Expedia Plugin and will try to pull information about Flights & it will show the information

SerpApi is a real-time API to access Google search results.

#### Wikipedia and llm-math tool

In [ ]:
!pip install wikipedia

In [ ]:
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain.llms import OpenAI

In [ ]:
llm = OpenAI(temperature=0)

In [ ]:
# install this package: pip install wikipedia

# The tools we'll give the Agent access to. Note that the 'llm-math' tool uses an LLM, so we need to pass that in.
tools = load_tools(["wikipedia", "llm-math"], llm=llm)

# Finally, let's initialize an agent with the tools, the language model, and the type of agent we want to use.
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Let's test it out!


agent.run("What was the GDP of US in 2024?")

 I should use Wikipedia to find the answer
Action: wikipedia
Action Input: GDP of US in 2024
Observation: Page: List of U.S. states and territories by GDP
Summary: This is a list of U.S. states and territories by gross domestic product (GDP). This article presents the 50 U.S. states and the District of Columbia and their nominal GDP at current prices.
The data source for the list is the Bureau of Economic Analysis (BEA) in 2024. The BEA defined GDP by state as "the sum of value added from all industries in the state."
Nominal GDP does not take into account differences in the cost of living in different countries, and the results can vary greatly from one year to another based on fluctuations in the exchange rates of the country's currency. Such fluctuations may change a country's ranking from one year to the next, even though they often make little or no difference in the standard of living of its population.
Overall, in the calendar year 2024, the United States' Nominal GDP at Current

'The GDP of US in 2024 was $28.269 trillion.'

##**07: Memory**

Chatbot application like ChatGPT, you will notice that it remember past information

In [ ]:
from langchain.llms import OpenAI

llm = OpenAI(temperature=0.9)

In [ ]:
from langchain.prompts import PromptTemplate

prompt_template_name = PromptTemplate(
    input_variables =['cuisine'],
    template = "I want to open a restaurant for {cuisine} food. Suggest a fency name for this."
)

In [ ]:
from langchain.chains import LLMChain

chain = LLMChain(llm=llm,prompt=prompt_template_name)
name = chain.run("Mexican")
print(name)



1. "La Cantina de México"
2. "El Sabor Auténtico"
3. "Casa de la Comida Mexicana"
4. "Viva México Cocina"
5. "The Mexican Kitchen"
6. "Fiesta Mexicana"
7. "Sabores de México"
8. "El Rincón Mexicano"
9. "La Cocina de Abuela"
10. "La Hacienda Mexicana"


In [ ]:
name = chain.run("Indian")
print(name)



1. "Spice Kingdom"
2. "Tandoori House"
3. "Curry Palace"
4. "Naan Stop"
5. "Masala Magic"
6. "Saffron Bazaar"
7. "Mughlai Delight"
8. "Chai & Chaat"
9. "Punjabi Grill"
10. "The Bollywood Bistro"


In [ ]:
chain.memory

In [ ]:
type(chain.memory)

NoneType

##**ConversationBufferMemory**

We can attach memory to remember all previous conversation

In [ ]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()

chain = LLMChain(llm=llm, prompt=prompt_template_name, memory=memory)
name = chain.run("Mexican")
print(name)



"Cantina de Oro" (meaning "Gold Cantina")


In [ ]:
name = chain.run("Arabic")
print(name)



"Al-Fawwarah" (The Oasis)


In [ ]:
print(chain.memory.buffer)

Human: Mexican
AI: 

"Cantina de Oro" (meaning "Gold Cantina")
Human: Arabic
AI: 

"Al-Fawwarah" (The Oasis)


##**ConversationChain**

Conversation buffer memory goes growing endlessly

Just remember last 5 Conversation Chain

Just remember last 10-20 Conversation Chain

In [ ]:
from langchain.chains import ConversationChain

convo = ConversationChain(llm=OpenAI(temperature=0.7))
print(convo.prompt.template)

The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Human: {input}
AI:


In [ ]:
convo.run("Who won the first cricket world cup?")

" The first cricket world cup was won by India in 1983. The final match took place at Lord's Cricket Ground in London, and India beat the West Indies by 43 runs. The Indian cricket team was captained by Kapil Dev, and Mohinder Amarnath was awarded the Man of the Match for his all-round performance. This victory was a major upset as the West Indies had won the previous two world cups and were considered unbeatable at the time. Do you have any other questions about the cricket world cup?"

In [ ]:
convo.run("How much is 5+5?")

'  5+5 is equal to 10. Do you have any other questions?'

In [ ]:
convo.run("Who was the captain of the winning team?")

' The captain of the winning team, India, was Kapil Dev. Do you have any other questions?'

In [ ]:
print(convo.memory.buffer)

Human: Who won the first cricket world cup?
AI:  The first cricket world cup was won by India in 1983. The final match took place at Lord's Cricket Ground in London, and India beat the West Indies by 43 runs. The Indian cricket team was captained by Kapil Dev, and Mohinder Amarnath was awarded the Man of the Match for his all-round performance. This victory was a major upset as the West Indies had won the previous two world cups and were considered unbeatable at the time. Do you have any other questions about the cricket world cup?
Human: How much is 5+5?
AI:   5+5 is equal to 10. Do you have any other questions?
Human: Who was the captain of the winning team?
AI:  The captain of the winning team, India, was Kapil Dev. Do you have any other questions?


##**ConversationBufferWindowMemory**

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=3)

convo = ConversationChain(
    llm=OpenAI(temperature=0.7),
    memory=memory
)
convo.run("Who won the first cricket world cup?")

" The first cricket world cup was won by the West Indies in 1975. The final match was played between West Indies and Australia at Lord's Cricket Ground in London, England. West Indies won by 17 runs."

In [ ]:
convo.run("How much is 5+5?")

'  5+5 is equal to 10.'

In [ ]:
convo.run("Who was the captain of the winning team?")

' The captain of the West Indies team during the 1975 cricket world cup was Clive Lloyd. He was known for his aggressive batting and leadership skills, and he led the team to victory in the first two cricket world cups in 1975 and 1979.'

In [ ]:
print(convo.memory.buffer)

Human: How much is 5+5?
AI:   5+5 is equal to 10.
Human: Who was the captain of the winning team?
AI:  The captain of the West Indies team during the 1975 cricket world cup was Clive Lloyd. He was known for his aggressive batting and leadership skills, and he led the team to victory in the first two cricket world cups in 1975 and 1979.
Human: Who was the captain of the winning team?
AI:  The captain of the West Indies team during the 1975 cricket world cup was Clive Lloyd. He was known for his aggressive batting and leadership skills, and he led the team to victory in the first two cricket world cups in 1975 and 1979.


#**08: Document Loaders**


In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 6.8 MB/s eta 0:00:00


In [ ]:
from langchain.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/my_paper.pdf")
pages = loader.load()

In [ ]:
pages

[Document(metadata={'source': '/content/my_paper.pdf', 'page': 0}, page_content='See discussions, st ats, and author pr ofiles f or this public ation at : https://www .researchgate.ne t/public ation/357213035\nDevelopment of Multiple Combined Regression Methods for Rainfall\nMeasu rement Development of Multiple Combined Regression Methods for\nRainfall Measu rement\nArticle  · Dec ember 2021\nCITATIONS\n0READS\n711\n6 author s, including:\nNusr at Jahan Pr ottasha\nDaff odil Int ernational Univ ersity\n26 PUBLICA TIONS \xa0\xa0\xa0299 CITATIONS \xa0\xa0\xa0\nSEE PROFILE\nMd K owsher\nStevens Instit ute of T echnolog y\n73 PUBLICA TIONS \xa0\xa0\xa0561 CITATIONS \xa0\xa0\xa0\nSEE PROFILE\nRokeya Khat un Shorna\nJahangirnag ar Univ ersity\n6 PUBLICA TIONS \xa0\xa0\xa05 CITATIONS \xa0\xa0\xa0\nSEE PROFILE\nNiaz Mur shed\nJahangirnag ar Univ ersity\n3 PUBLICA TIONS \xa0\xa0\xa00 CITATIONS \xa0\xa0\xa0\nSEE PROFILE\nAll c ontent f ollo wing this p age was uplo aded b y Niaz Mur shed  on 21 